# 🧠 Module 2: Review Sentiment Quality (DistilBERT)
**ECommerce Product Trust Assessor**

Fine-tunes DistilBERT on Amazon review sentiment for the Trust Assessor.

**Runtime:** GPU (T4 recommended on Colab)

**Output:** `models/sentiment/distilbert_finetuned/`

In [ ]:
!pip install -q transformers datasets accelerate torch scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# ── Load & Prepare Data ──────────────────────────────────────────────────────
raw = load_dataset('McAuley-Lab/Amazon-Reviews-2023', 'raw_review_Electronics',
                   split='full', trust_remote_code=True)
df = raw.to_pandas()[['text', 'rating']].dropna().sample(20_000, random_state=42)

# Convert ratings to binary sentiment (1–2 = negative, 4–5 = positive, drop 3)
df = df[df['rating'] != 3].copy()
df['label'] = (df['rating'] >= 4).astype(int)

print(f'Dataset: {df.shape}')
print(df['label'].value_counts())

In [ ]:
# ── Tokenize ─────────────────────────────────────────────────────────────────
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_df, val_df = df[:16000], df[16000:]

train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
print('✅ Tokenization complete')

In [ ]:
# ── Fine-tune DistilBERT ──────────────────────────────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir='./models/sentiment/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# ── Save Fine-tuned Model ────────────────────────────────────────────────────
import os
os.makedirs('models/sentiment/distilbert_finetuned', exist_ok=True)

model.save_pretrained('models/sentiment/distilbert_finetuned')
tokenizer.save_pretrained('models/sentiment/distilbert_finetuned')

print('✅ Fine-tuned DistilBERT saved to models/sentiment/distilbert_finetuned/')
print('Evaluation:', trainer.evaluate())